# JIN AI — Phase 18: train tiếp từ Phase 17 tới khi loss ngừng giảm

Phase 17 dừng đúng budget 100M token: **best val 3.8585** (ppl 47.39) @ step 2500,
val cuối 4.1612. Model 124M tham số mà mới thấy 100M token = **0.8 token/tham số**
(GPT-2 Small gốc: ~80 token/tham số), nên loss còn giảm nhiều.

Notebook này chạy **300M token nữa** trên đúng corpus v7 (bin train đã cap 300M token),
khởi đầu từ `run_100m/best.pt`, cosine 1e-4 → 1e-5, và tự dừng an toàn theo thời gian
để phiên sau resume đúng chỗ.

## Chuẩn bị (làm 1 lần)
1. Notebook Phase 17 → **Save Version** (output đã có `phase17_output/run_100m/best.pt`
   và `repo/data/gpt2_corpus_v7`).
2. Notebook này: **Add Input → Your Work → version Phase 17 đó**. Thiếu nó thì không có
   checkpoint để bắt đầu, notebook dừng ngay.
3. **Settings → Internet = On** (để clone code), **Accelerator = GPU T4 x2**.

Thời gian: 300M token ở ~7.300 tok/s thực đo ≈ **11 giờ**. Giới hạn phiên là 12h nên
Stage G đặt `--session-max-minutes 560` để tự lưu rồi thoát; phiên sau chỉ chạy Stage 1
rồi Stage G2 (resume), `total_steps` không đổi.

In [ ]:
import glob, json, os, shutil, subprocess, sys, zipfile

WORK = "/kaggle/working/repo"
OUT = "/kaggle/working/phase18_output"
os.makedirs(OUT, exist_ok=True)

def _first(pattern, root="/kaggle/input"):
    hits = sorted(glob.glob(os.path.join(root, "**", pattern), recursive=True))
    return hits[0] if hits else None

# ---- code: clone repo (bản mới nhất), nếu không có mạng thì dùng bản trong dataset ----
REPO_URL = "https://github.com/Annguyen0410/JIN-AI-LLM-from-zero-to-hero-ZHIEND.git"
if os.path.exists(WORK):
    shutil.rmtree(WORK)
cloned = False
try:
    subprocess.run(["git", "clone", "--depth", "1", REPO_URL, WORK],
                   check=True, capture_output=True, timeout=300)
    cloned = os.path.exists(os.path.join(WORK, "train_gpt2.py"))
except Exception as exc:
    print("clone không chạy được (Internet off?) -> dùng bản trong dataset:", exc)

if not cloned:
    entry = _first("train_gpt2.py")
    zipped = _first("*.zip")
    if entry:
        shutil.copytree(os.path.dirname(entry), WORK, ignore=shutil.ignore_patterns(
            ".git", "__pycache__", "*.pyc", "data", "checkpoints", "cache"))
    elif zipped:
        os.makedirs(WORK, exist_ok=True)
        zipfile.ZipFile(zipped).extractall(WORK)
        inner = [d for d in os.listdir(WORK)
                 if os.path.isfile(os.path.join(WORK, d, "train_gpt2.py"))]
        if inner:
            WORK = os.path.join(WORK, inner[0])
    print("code: dùng bản trong input (có thể cũ hơn)")

os.chdir(WORK)
CODE_REV = subprocess.run(["git", "rev-parse", "HEAD"], capture_output=True,
                          text=True).stdout.strip() or "unknown"
print("code revision:", CODE_REV)

# ---- điểm khởi đầu: best.pt của Phase 17 ----
PREV = (_first("run_100m/best.pt")
        or _first("phase17_output/**/best.pt")
        or ("/kaggle/working/phase17_output/run_100m/best.pt"
            if os.path.exists("/kaggle/working/phase17_output/run_100m/best.pt") else None))
if not PREV:
    raise SystemExit("KHÔNG thấy run_100m/best.pt của Phase 17 — Add Input output của "
                     "version đã train 100M (Stage G) rồi chạy lại cell này.")

PREV_MAN = PREV + ".manifest.json"
if os.path.exists(PREV_MAN):
    man = json.load(open(PREV_MAN, encoding="utf-8"))
    print("prev:", {k: man.get(k) for k in
                    ("iteration", "best_val_loss", "additional_tokens_this_run",
                     "parent_sha256", "init_mode")})
    assert man.get("additional_tokens_this_run", 0) > 50_000_000, \
        "best.pt này chưa train 100M token — không phải output Stage G của Phase 17"

parquet = _first("*.parquet")
if not parquet:
    raise SystemExit("KHÔNG thấy *.parquet — Add Input dataset Wikipedia (chỉ cần khi phải "
                     "dựng lại corpus).")
PARQUET_DIR = os.path.dirname(parquet)
SOURCE_DIR = os.path.basename(PARQUET_DIR.rstrip("/"))
SOURCE_ID = SOURCE_DIR.replace("-", "_")

CONFIG = {
    "prev_checkpoint": PREV,
    "parquet_dir": PARQUET_DIR,
    "reuse_corpus": _first("gpt2_corpus_v7/manifest.json"),
    "source_dir": SOURCE_DIR,
    "source_id": SOURCE_ID,
    "license_map_path": "/kaggle/working/license_map.json",
    "out_dir": OUT,
    "code_revision": CODE_REV,
}
print(json.dumps(CONFIG, indent=2, ensure_ascii=False))
print("checkpoint:", round(os.path.getsize(PREV) / 1e9, 2), "GB")
print(subprocess.run(["df", "-h", "/kaggle/working"], capture_output=True, text=True).stdout)

## Stage 1. Corpus v7 — dùng lại nếu có, không thì dựng lại

Phase 17 đã cap bin train ở **300M token**; Phase 18 dùng đúng phần đó nên không cần
ingest thêm. Nếu bạn không attach `gpt2_corpus_v7` (hoặc nó không nằm trong output đã
Save), cell này dựng lại với **số doc cố định** để corpus tái lập được giữa các phiên.

In [ ]:
LICENSE_TEXT = "CC-BY-SA-3.0 and GFDL as dataset card; review upstream terms"
SOURCE_URL = "https://huggingface.co/datasets/wikimedia/wikipedia"
FIXED_DOCS = 240_000          # cố định để corpus tái lập, không phụ thuộc dung lượng trống
MAX_TRAIN_TOKENS = "300000000"
CURATED = "data/curated_v7"
TOKENS = "data/gpt2_corpus_v7"

if CONFIG["reuse_corpus"]:
    shutil.rmtree(TOKENS, ignore_errors=True)
    shutil.copytree(os.path.dirname(CONFIG["reuse_corpus"]), TOKENS)
    print("corpus: dùng lại từ input ->", TOKENS)
else:
    license_map = {SOURCE_DIR: {
        "license": LICENSE_TEXT, "source_url": SOURCE_URL,
        "policy_source": "wikipedia-en-20231101",
        "revision": "b04c8d1ceb2f5cd4588862100d08de323dccfbaa",
        "text_columns": ["text"], "language": "en",
    }}
    json.dump(license_map, open(CONFIG["license_map_path"], "w", encoding="utf-8"), indent=2)

    subprocess.run([sys.executable, "scripts/ingest_corpus.py",
                    "--input-root", PARQUET_DIR, "--source", SOURCE_ID,
                    "--license-map", CONFIG["license_map_path"],
                    "--max-docs-per-source", str(FIXED_DOCS),
                    "--out-root", CURATED]).check_returncode()
    subprocess.run([sys.executable, "scripts/verify_corpus.py",
                    "--in-root", CURATED, "--out-root", CURATED,
                    "--pii-policy", "exclude", "--license-policy", "enforce"]).check_returncode()

    raw = json.load(open(f"{CURATED}/raw_manifest.json", encoding="utf-8"))["sources"]
    available = [s["source_id"] for s in raw if s.get("present") and s.get("documents")]
    print("sources:", available)
    MIX = ",".join(f"{sid}:{1.0 / len(available):.2f}" for sid in available)
    subprocess.run([sys.executable, "scripts/build_tokenized_corpus.py",
                    "--splits", f"{CURATED}/splits", "--out", TOKENS,
                    "--max-train-tokens", MAX_TRAIN_TOKENS,
                    "--mix", MIX]).check_returncode()

tok_man = json.load(open(f"{TOKENS}/manifest.json", encoding="utf-8"))
print(json.dumps({k: tok_man.get(k) for k in ("train", "val", "test")}, indent=2)[:900])
TRAIN_DATA = TOKENS
TOKENS_PER_STEP = 4 * 8 * 1024
n_train = tok_man["train"].get("tokens") or tok_man["train"].get("num_tokens")
if n_train:
    print(f"train bin: {n_train:,} token = {n_train / TOKENS_PER_STEP:,.0f} step")

## Stage 2. Smoke 20 step từ `best.pt`

Không phải để đo chất lượng — chỉ để chứng minh weights của Phase 17 đã vào model:
loss phải **< 4.8** (random init ≈ 10.8; Phase 17 để lại ~3.9).

In [ ]:
# Smoke: 20 step từ best.pt của Phase 17. Đây là bước TRAIN thật, không chỉ đọc file.
smoke_dir = f"{OUT}/smoke"
shutil.rmtree(smoke_dir, ignore_errors=True)   # output cũ không được che lỗi của lần này
subprocess.run([sys.executable, "train_gpt2.py",
                "--preset", "gpt2_small_reference", "--data-dir", TRAIN_DATA,
                "--init-from", CONFIG["prev_checkpoint"], "--continue-pretrain",
                "--out-dir", smoke_dir, "--seq-len", "1024",
                "--batch-size", "4", "--grad-accum", "4",
                "--max-steps", "20", "--warmup", "2", "--lr", "1e-4",
                "--eval-interval", "10", "--eval-iters", "4",
                "--save-interval", "10", "--seed", "17"]).check_returncode()

# Lineage nằm trong manifest, KHÔNG nằm trong .pt: payload checkpoint không có
# trường init_mode, nên đọc .pt rồi blob.get("init_mode") luôn trả None.
smoke_man_path = f"{smoke_dir}/latest.pt.manifest.json"
assert os.path.exists(smoke_man_path), f"smoke không ghi manifest: {smoke_man_path}"
smoke_man = json.load(open(smoke_man_path, encoding="utf-8"))
prev_man = json.load(open(PREV_MAN, encoding="utf-8")) if os.path.exists(PREV_MAN) else {}
print("smoke best_val_loss:", round(smoke_man["best_val_loss"], 4),
      "| init_mode:", smoke_man.get("init_mode"),
      "| parent:", str(smoke_man.get("parent_sha256"))[:12])
assert smoke_man["best_val_loss"] < 4.8, "weights Phase 17 không được load đúng"
assert smoke_man.get("init_mode") == "continued_pretrain", smoke_man.get("init_mode")
if prev_man.get("sha256"):
    assert smoke_man.get("parent_sha256") == prev_man["sha256"], \
        "smoke không init từ đúng best.pt của Phase 17"

# dọn .pt của smoke để lấy lại ~1.5 GB
for p in glob.glob(f"{smoke_dir}/*.pt"):
    os.remove(p)


## G. Train tiếp — 300M token (9.155 step)

`--init-from best.pt` = **experiment mới**: optimizer/schedule mới, `best_val` riêng,
manifest ghi `parent_sha256` = best.pt của Phase 17. Không dùng `--resume-from` vì resume
đòi fingerprint trùng tuyệt đối (kể cả `code_revision`); chỉ cần một dòng code đổi là bị
từ chối — đúng thiết kế, nhưng chặn việc train tiếp qua nhiều phiên có sửa code.

`--session-max-minutes 560` chỉ tạm dừng, **không** đổi `total_steps`.

In [ ]:
CHOSEN_LR = "1e-4"          # giữ như Phase 17: cùng recipe, chỉ dài hơi hơn
TARGET_TOKENS = 300_000_000
TOTAL_STEPS = TARGET_TOKENS // TOKENS_PER_STEP
print(f"{TOTAL_STEPS:,} step x {TOKENS_PER_STEP:,} = "
      f"{TOTAL_STEPS * TOKENS_PER_STEP / 1e6:.0f}M token")
if n_train:
    print(f"corpus {n_train:,} token -> {TOTAL_STEPS * TOKENS_PER_STEP / n_train:.2f} vòng; "
          "val đo mỗi 500 step nên best.pt giữ mốc tốt nhất")

subprocess.run([sys.executable, "train_gpt2.py",
                "--preset", "gpt2_small_reference", "--data-dir", TRAIN_DATA,
                "--init-from", CONFIG["prev_checkpoint"], "--continue-pretrain",
                "--out-dir", f"{OUT}/run_400m", "--seq-len", "1024",
                "--batch-size", "4", "--grad-accum", "8",
                "--max-steps", str(TOTAL_STEPS), "--warmup", "300", "--lr", CHOSEN_LR,
                "--eval-interval", "500", "--eval-iters", "20",
                "--save-interval", "250", "--session-max-minutes", "560",
                "--seed", "17"]).check_returncode()

## G2. Phiên sau — resume đúng chỗ đã dừng

Save Version → mở phiên mới → Add Input output của version vừa train → chạy Stage 1
(clone) rồi cell này. Không truyền `--init-from`, chỉ `--resume-from`: không thấy
checkpoint thì nó **dừng**, không train lại từ đầu.

Điều kiện: code revision phải giống phiên trước. Notebook clone `main`, nên đừng sửa
`train_gpt2.py` giữa chừng; nếu buộc phải sửa, ghim commit trong cell clone.

In [ ]:
PREV_RUN = (_first("run_400m/latest.pt")
            or ("/kaggle/working/phase18_output/run_400m/latest.pt"
                if os.path.exists("/kaggle/working/phase18_output/run_400m/latest.pt") else None))
if PREV_RUN and os.path.exists(PREV_RUN):
    man = json.load(open(PREV_RUN + ".manifest.json", encoding="utf-8"))
    print("resume từ step", man.get("iteration"), "| best val", man.get("best_val_loss"))
    subprocess.run([sys.executable, "train_gpt2.py",
                    "--preset", "gpt2_small_reference", "--data-dir", TRAIN_DATA,
                    "--resume-from", PREV_RUN, "--out-dir", f"{OUT}/run_400m",
                    "--seq-len", "1024", "--batch-size", "4", "--grad-accum", "8",
                    "--max-steps", str(TOTAL_STEPS), "--warmup", "300", "--lr", CHOSEN_LR,
                    "--eval-interval", "500", "--eval-iters", "20",
                    "--save-interval", "250", "--session-max-minutes", "560",
                    "--seed", "17"]).check_returncode()
else:
    print("Không có latest.pt của run_400m — chưa chạy Stage G hoặc chưa attach output.")

## H. Báo cáo — so với mốc Phase 17

Kiểm tra SHA khớp manifest rồi so `best_val_loss` với 3.8585. Đây là ppl trên `val.bin`
của chính corpus này, không phải phát biểu về năng lực.

In [ ]:
import hashlib

def sha(path):
    h = hashlib.sha256()
    with open(path, "rb") as fh:
        for chunk in iter(lambda: fh.read(1 << 22), b""):
            h.update(chunk)
    return h.hexdigest()

PHASE17_BEST_VAL = 3.8585
run_dir = f"{OUT}/run_400m"
for name in ("latest.pt", "best.pt"):
    path = f"{run_dir}/{name}"
    if not os.path.exists(path):
        continue
    man = json.load(open(path + ".manifest.json", encoding="utf-8"))
    assert sha(path) == man["sha256"], f"{name}: hash không khớp manifest"
    print(f"{name}: sha256={man['sha256'][:16]}… step={man['iteration']} "
          f"val={man['best_val_loss']:.4f} tokens={man['additional_tokens_this_run']:,} "
          f"parent={man['parent_sha256'][:12]}… mode={man['init_mode']}")

best = json.load(open(f"{run_dir}/best.pt.manifest.json", encoding="utf-8"))
delta = PHASE17_BEST_VAL - best["best_val_loss"]
print(f"\nbest mới {best['best_val_loss']:.4f} vs Phase 17 {PHASE17_BEST_VAL:.4f} "
      f"-> {'giảm' if delta > 0 else 'tăng'} {abs(delta):.4f}")
print("Nếu val vẫn giảm ở step cuối thì tăng TARGET_TOKENS cùng với dữ liệu mới, "
      "đừng chỉ kéo dài cosine trên dữ liệu cũ.")

## Muốn loss thấp hơn nữa

Loss bị chặn bởi **dữ liệu**, không bởi số step. Corpus v7 là wiki-only: cap 300M token
nhưng thực tế chỉ dựng được **160,9M token** train, nên Phase 18 chạy 300M token =
**1,86 vòng** trên đúng 160,9M đó (100M đầu đã dùng ở Phase 17). Ba hướng, xếp theo hiệu
quả trên mỗi giờ GPU:

1. **Thêm dữ liệu mới** — ingest thêm Wikipedia (11 GB ≈ 2,2B token) và nâng
   `--max-train-tokens` lên 800M–1B. Đây là hướng giảm loss thật.
2. **Mixture** — thêm nguồn thứ hai (code/math) vào `--mix`; wiki-only thì ppl Wikipedia
   giảm nhưng chat/tool vẫn 0%.
3. **Model lớn hơn** — 124M đang ở ~0,8 token/tham số; tăng tham số chỉ có lợi khi đã có
   ≥20 tỷ token. Chưa nên.

Nói thẳng: benchmark task hiện 1/73. Pretraining làm ppl giảm, nhưng nó **không** tự sinh
ra khả năng làm bài — phần đó là SFT/instruction data, thuộc phase sau.